# Migration: Dept_pkg Conversion
**Source:** Dept_pkg ODI Session
**Target:** Databricks Spark SQL / Delta Lake
**Date:** 2024-05-22

In [ ]:
dbutils.widgets.text("ODI_SESS_NO", "f0247733-ea2b-4de3-b73d-c84d0fca5168")
dbutils.widgets.text("DATASOURCE_NUM_ID", "")
dbutils.widgets.text("V_Dept", "")

### ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_dept_param AS
SELECT 
    COALESCE(
        (SELECT date_format(last_run_dt, 'dd-MM-yy') 
         FROM workspace.odi_trg.log_table1 
         WHERE pkg_name = 'Dept_pkg'),
        date_format(current_timestamp(), 'dd-MM-yy')
    ) AS v_dept_extract_date;

### Log Table Update (SCEN_TASK_NO 2)

In [ ]:
%sql
UPDATE workspace.odi_trg.log_table1 
SET last_run_dt = current_timestamp() 
WHERE pkg_name = 'Dept_pkg';

### Staging Table Preparation (SCEN_TASK_NO 30, 40)

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_trg.c_0filter;

CREATE TABLE workspace.odi_trg.c_0filter
(
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP
)
USING DELTA;

### Extract into Staging (SCEN_TASK_NO 50)

In [ ]:
%sql
INSERT INTO workspace.odi_trg.c_0filter
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT 
    SRC_DEPARTMENTS_1.DEPARTMENT_ID AS DEPARTMENT_ID,
    SRC_DEPARTMENTS_1.DEPARTMENT_NAME AS DEPARTMENT_NAME,
    SRC_DEPARTMENTS_1.MANAGER_ID AS MANAGER_ID,
    SRC_DEPARTMENTS_1.LOCATION_ID AS LOCATION_ID,
    SRC_DEPARTMENTS_1.LAST_UPD_DT AS LAST_UPD_DT
FROM workspace.odi_src.src_departments AS SRC_DEPARTMENTS_1
WHERE (1=1)
  AND (SRC_DEPARTMENTS_1.LAST_UPD_DT >= to_date((SELECT v_dept_extract_date FROM v_dept_param), 'dd-MM-yy'));

### Staging Stats (SCEN_TASK_NO 60)

In [ ]:
%sql
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.c_0filter ZORDER BY (DEPARTMENT_ID);

### Flow Table Preparation (SCEN_TASK_NO 80, 90)

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;

CREATE TABLE workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID       BIGINT,
    DEPARTMENT_NAME     STRING,
    MANAGER_ID          BIGINT,
    LOCATION_ID         BIGINT,
    LAST_UPD_DT         TIMESTAMP,
    IND_UPDATE          STRING
)
USING DELTA;

### Delta Calculation for Flow Table (SCEN_TASK_NO 100)

In [ ]:
%sql
INSERT INTO workspace.odi_trg.i_trg_departments_flow
(
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT,
    IND_UPDATE
)
SELECT 
    S.DEPARTMENT_ID,
    S.DEPARTMENT_NAME,
    S.MANAGER_ID,
    S.LOCATION_ID,
    S.LAST_UPD_DT,
    S.IND_UPDATE
FROM (
    SELECT 
        FILTER_A.DEPARTMENT_ID,
        FILTER_A.DEPARTMENT_NAME,
        FILTER_A.MANAGER_ID,
        FILTER_A.LOCATION_ID,
        FILTER_A.LAST_UPD_DT,
        'I' AS IND_UPDATE
    FROM workspace.odi_trg.c_0filter AS FILTER_A
) AS S
WHERE NOT EXISTS (
    SELECT 1 FROM workspace.odi_trg.trg_departments AS T
    WHERE T.DEPARTMENT_ID = S.DEPARTMENT_ID 
      AND ((T.DEPARTMENT_NAME = S.DEPARTMENT_NAME) OR (T.DEPARTMENT_NAME IS NULL AND S.DEPARTMENT_NAME IS NULL))
      AND ((T.MANAGER_ID = S.MANAGER_ID) OR (T.MANAGER_ID IS NULL AND S.MANAGER_ID IS NULL))
      AND ((T.LOCATION_ID = S.LOCATION_ID) OR (T.LOCATION_ID IS NULL AND S.LOCATION_ID IS NULL))
      AND ((T.LAST_UPD_DT = S.LAST_UPD_DT) OR (T.LAST_UPD_DT IS NULL AND S.LAST_UPD_DT IS NULL))
);

### Optimize Flow Table (SCEN_TASK_NO 110, 120)

In [ ]:
%sql
SET spark.databricks.delta.optimize.zorder.checkStatsCollection.enabled = false;
OPTIMIZE workspace.odi_trg.i_trg_departments_flow ZORDER BY (DEPARTMENT_ID);

### Error Tables Setup (SCEN_TASK_NO 130, 150)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.odi_trg.snp_check_tab
(
    CATALOG_NAME    STRING,
    SCHEMA_NAME     STRING,
    RESOURCE_NAME   STRING,
    FULL_RES_NAME   STRING,
    ERR_TYPE        STRING,
    ERR_MESS        STRING,
    CHECK_DATE      TIMESTAMP,
    ORIGIN          STRING,
    CONS_NAME       STRING,
    CONS_TYPE       STRING,
    ERR_COUNT       BIGINT
) USING DELTA;

CREATE TABLE IF NOT EXISTS workspace.odi_trg.e_trg_departments
(
    ODI_ROW_ID      STRING,
    ODI_ERR_TYPE    STRING,
    ODI_ERR_MESS    STRING,
    ODI_CHECK_DATE  TIMESTAMP,
    DEPARTMENT_ID   BIGINT,
    DEPARTMENT_NAME STRING,
    MANAGER_ID      BIGINT,
    LOCATION_ID     BIGINT,
    LAST_UPD_DT     TIMESTAMP,
    ODI_ORIGIN      STRING,
    ODI_CONS_NAME   STRING,
    ODI_CONS_TYPE   STRING,
    ODI_PK          STRING,
    ODI_SESS_NO     STRING
) USING DELTA;

### Error Cleanup (SCEN_TASK_NO 140, 160)

In [ ]:
%sql
DELETE FROM workspace.odi_trg.snp_check_tab
WHERE SCHEMA_NAME = 'ODI_TRG'
  AND ORIGIN = '(7)test.Dept_MAp'
  AND ERR_TYPE = 'F';

DELETE FROM workspace.odi_trg.e_trg_departments
WHERE (ODI_ERR_TYPE = 'S' AND 1 = 0)
   OR (ODI_ERR_TYPE = 'F' AND ODI_ORIGIN = '(7)test.Dept_MAp');

### Primary Key Check (SCEN_TASK_NO 180)

In [ ]:
%sql
INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_ORIGIN,
    ODI_CHECK_DATE,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT 
    uuid(),
    '${ODI_SESS_NO}', 
    CAST(monotonically_increasing_id() AS STRING),
    'F', 
    'ODI-15064: The primary key PK_department_id is not unique.',
    '(7)test.Dept_MAp',
    current_timestamp(),
    'PK_department_id',
    'PK',	
    TRG_DEPARTMENTS.DEPARTMENT_ID,
    TRG_DEPARTMENTS.DEPARTMENT_NAME,
    TRG_DEPARTMENTS.MANAGER_ID,
    TRG_DEPARTMENTS.LOCATION_ID,
    TRG_DEPARTMENTS.LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow AS TRG_DEPARTMENTS
WHERE EXISTS (
    SELECT 1
    FROM workspace.odi_trg.i_trg_departments_flow AS SUB
    WHERE SUB.DEPARTMENT_ID = TRG_DEPARTMENTS.DEPARTMENT_ID
    GROUP BY SUB.DEPARTMENT_ID
    HAVING COUNT(1) > 1
);

### Not Null Check (SCEN_TASK_NO 190)

In [ ]:
%sql
INSERT INTO workspace.odi_trg.e_trg_departments
(
    ODI_PK,
    ODI_SESS_NO,
    ODI_ROW_ID,
    ODI_ERR_TYPE,
    ODI_ERR_MESS,
    ODI_CHECK_DATE,
    ODI_ORIGIN,
    ODI_CONS_NAME,
    ODI_CONS_TYPE,
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
)
SELECT
    uuid(),
    '${ODI_SESS_NO}', 
    CAST(monotonically_increasing_id() AS STRING),
    'F', 
    'ODI-15066: The column DEPARTMENT_NAME cannot be null.',
    current_timestamp(),
    '(7)test.Dept_MAp',
    'DEPARTMENT_NAME',
    'NN',	
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
FROM workspace.odi_trg.i_trg_departments_flow
WHERE DEPARTMENT_NAME IS NULL;

### Remove Failed Records from Flow (SCEN_TASK_NO 210)

In [ ]:
%sql
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING workspace.odi_trg.e_trg_departments AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
AND S.ODI_SESS_NO = '${ODI_SESS_NO}'
WHEN MATCHED THEN DELETE;

### Error Audit Summary (SCEN_TASK_NO 220)

In [ ]:
%sql
INSERT INTO workspace.odi_trg.snp_check_tab
(
    SCHEMA_NAME,
    RESOURCE_NAME,
    FULL_RES_NAME,
    ERR_TYPE,
    ERR_MESS,
    CHECK_DATE,
    ORIGIN,
    CONS_NAME,
    CONS_TYPE,
    ERR_COUNT
)
SELECT	
    'ODI_TRG',
    'TRG_DEPARTMENTS',
    'workspace.odi_trg.trg_departments',
    E.ODI_ERR_TYPE,
    E.ODI_ERR_MESS,
    E.ODI_CHECK_DATE,
    E.ODI_ORIGIN,
    E.ODI_CONS_NAME,
    E.ODI_CONS_TYPE,
    count(1) 
FROM workspace.odi_trg.e_trg_departments AS E
WHERE E.ODI_ERR_TYPE = 'F'
  AND E.ODI_ORIGIN = '(7)test.Dept_MAp'
GROUP BY 
    E.ODI_ERR_TYPE,
    E.ODI_ERR_MESS,
    E.ODI_CHECK_DATE,
    E.ODI_ORIGIN,
    E.ODI_CONS_NAME,
    E.ODI_CONS_TYPE;

### Flag Records for Update (SCEN_TASK_NO 230)

In [ ]:
%sql
MERGE INTO workspace.odi_trg.i_trg_departments_flow AS T
USING workspace.odi_trg.trg_departments AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED THEN UPDATE SET T.IND_UPDATE = 'U';

### Final Merge into Target (SCEN_TASK_NO 270, 280)

In [ ]:
%sql
MERGE INTO workspace.odi_trg.trg_departments AS T
USING workspace.odi_trg.i_trg_departments_flow AS S
ON T.DEPARTMENT_ID = S.DEPARTMENT_ID
WHEN MATCHED AND S.IND_UPDATE = 'U' THEN UPDATE SET
    T.DEPARTMENT_NAME = S.DEPARTMENT_NAME,
    T.MANAGER_ID      = S.MANAGER_ID,
    T.LOCATION_ID     = S.LOCATION_ID,
    T.LAST_UPD_DT     = S.LAST_UPD_DT
WHEN NOT MATCHED AND S.IND_UPDATE = 'I' THEN INSERT (
    DEPARTMENT_ID,
    DEPARTMENT_NAME,
    MANAGER_ID,
    LOCATION_ID,
    LAST_UPD_DT
) VALUES (
    S.DEPARTMENT_ID,
    S.DEPARTMENT_NAME,
    S.MANAGER_ID,
    S.LOCATION_ID,
    S.LAST_UPD_DT
);

### Cleanup (SCEN_TASK_NO 340, 360)

In [ ]:
%sql
DROP TABLE IF EXISTS workspace.odi_trg.c_0filter;
DROP TABLE IF EXISTS workspace.odi_trg.i_trg_departments_flow;